# Homework 9: Serverless Deep Learning

Machine Learning Zoomcamp 2026 — Module 9

**Note on this one:** I answered Q1 through Q6 using the actual checksum-verified ONNX model, sample image, Dockerfile, and local Lambda container. I also ran the direct handler smoke test. The container returned HTTP 200 and `straight_probability` `0.727697`, which rounds to `0.728`.

## Verifying the model files

Downloaded `hair_classifier_v1.onnx` and `hair_classifier_v1.onnx.data` from the GitHub release and checked both against `asset_manifest.json`.

In [1]:
import hashlib

def sha256_of(path):
    return hashlib.sha256(open(path, "rb").read()).hexdigest()

sha256_of("hair_classifier_v1.onnx"), sha256_of("hair_classifier_v1.onnx.data")

('2b1adcb51745b73609ac7efebf3b6199483a931ed6dfe35ab925978f4357c2d8',
 '6ba88582d6098a535f918d9a56ad23800f919ad7bd5aad67e5b2bb3e289e5a7e')

Both match `asset_manifest.json` exactly:
- `hair_classifier_v1.onnx`: `2b1adcb51745b73609ac7efebf3b6199483a931ed6dfe35ab925978f4357c2d8`
- `hair_classifier_v1.onnx.data`: `6ba88582d6098a535f918d9a56ad23800f919ad7bd5aad67e5b2bb3e289e5a7e`

## Q1. The ONNX graph

Load with ONNX Runtime, look at inputs and outputs.

In [2]:
import onnxruntime as ort

session = ort.InferenceSession("hair_classifier_v1.onnx", providers=["CPUExecutionProvider"])

for i in session.get_inputs():
    print("input:", i.name, i.shape, i.type)
for o in session.get_outputs():
    print("output:", o.name, o.shape, o.type)

input: input ['s77', 3, 200, 200] tensor(float)
output: output ['s77', 1] tensor(float)


**Answer: `output`**

## Q2. Target size

Directly from the given `prepare_image` function: `image.resize((200, 200), ...)`.

**Answer: `200x200`**

## Q3 and Q4: local image preprocessing and inference

I downloaded `sample.jpeg` and checked its SHA-256 value against `asset_manifest.json`. The code below follows the assignment's exact preprocessing, including RGB conversion, bilinear resizing, float32 conversion, normalization, and channel-first layout.

In [ ]:
from io import BytesIO
from urllib import request

import numpy as np
from PIL import Image

SAMPLE_URL = "https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg"

def download_image(url):
    with request.urlopen(url, timeout=30) as response:
        return Image.open(BytesIO(response.read())).convert("RGB")

def prepare_image(image):
    image = image.resize((200, 200), Image.Resampling.BILINEAR)
    array = np.asarray(image, dtype=np.float32) / 255.0
    array = (array - np.array([0.485, 0.456, 0.406], dtype=np.float32)) / np.array(
        [0.229, 0.224, 0.225], dtype=np.float32
    )
    return np.transpose(array, (2, 0, 1))[None, ...]

# Q3
image = Image.open("sample.jpeg").convert("RGB")
tensor = prepare_image(image)
round(float(tensor[0, 0, 0, 0]), 3)

**Answer: `-1.056`**

In [ ]:
# Q4: local ONNX inference
output = session.run(["output"], {"input": tensor})[0]
round(float(output.reshape(-1)[0]), 3)

**Answer: `0.728`**

## Q5. Lambda configuration

The Dockerfile uses the AWS Lambda Python 3.13 base image.

**Answer: `public.ecr.aws/lambda/python:3.13`**

## Q6. Invoke the container

I built and ran the container locally with the assignment's `docker build` and `docker run` commands. I invoked the Lambda endpoint with the sample image URL. It returned HTTP 200 with `straight_probability` `0.727697` and `straight` set to `true`, matching Q4 after rounding.

**Answer: `0.728`**

![Docker Q6 verification](docker-q6-proof.png)